**BBH**

**Generate perturbed versions**

In [42]:
import pandas as pd
import re
import os

# === Step 0: Paste your JSON array here directly ===
llm_output =  


# === Step 1: Load BBH CSV ===
bbh_csv_file = "bbh_sample_200.csv"    # original CSV with input and target columns
output_csv_file = "bbh_perturbed_with_options.csv"

df_bbh = pd.read_csv(bbh_csv_file)

# Add ID if not present
if "id" not in df_bbh.columns:
    df_bbh.insert(0, "id", range(1, len(df_bbh)+1))

# Function to extract Options block from 'input' column
def extract_options(text):
    match = re.search(r"Options:\s*(.*)", text, flags=re.DOTALL)
    if match:
        options_text = match.group(0).strip()  # include "Options:" label
        return options_text
    else:
        return ""

# Map original question text -> options
question_to_options = {}
for _, row in df_bbh.iterrows():
    original_question = row['input'].split("Options:")[0].strip()
    options_text = extract_options(row['input'])
    question_to_options[original_question] = options_text

# === Step 2: Process LLM JSON array ===
rows = []
for item in llm_output:
    orig_question_text = item.get("original", "").strip()
    options = question_to_options.get(orig_question_text, "")
    
    row = {
        "original": f"{item.get('original','')}\n{options}",
        "lexical": f"{item.get('invariant_lexical','')}\n{options}",
        "syntactic": f"{item.get('invariant_syntactic','')}\n{options}",
        "contextual": f"{item.get('invariant_contextual','')}\n{options}"
    }
    
    # Lookup target
    match_target = df_bbh[df_bbh['input'].str.contains(re.escape(orig_question_text))]
    if not match_target.empty:
        row["target"] = match_target.iloc[0]['target']
    else:
        row["target"] = ""
    
    rows.append(row)

df_new = pd.DataFrame(rows)

# === Step 3: Assign IDs and append to CSV ===
if os.path.exists(output_csv_file):
    df_existing = pd.read_csv(output_csv_file)
    start_id = df_existing["id"].max() + 1 if "id" in df_existing.columns else len(df_existing)+1
    df_new.insert(0, "id", range(start_id, start_id + len(df_new)))
    df_new.to_csv(output_csv_file, mode="a", index=False, header=False, encoding="utf-8")
else:
    df_new.insert(0, "id", range(1, len(df_new)+1))
    df_new.to_csv(output_csv_file, index=False, header=True, encoding="utf-8")

df_final = pd.read_csv(output_csv_file)
max_id = df_final["id"].max()
print(f"The CSV file now contains entries up to ID: {max_id}")


The CSV file now contains entries up to ID: 200


In [ ]:
import pandas as pd

# File to update
csv_file = "bbh_perturbed_with_options.csv"

# Introduction text to prepend
introduction_text = (
    "In the following sentences, explain the antecedent of the pronoun "
    "(which thing the pronoun refers to), or state that it is ambiguous.\n"
)

# Load CSV
df = pd.read_csv(csv_file)

# Columns to update
question_cols = ["original", "lexical", "syntactic", "contextual"]

# Prepend introduction to each column
for col in question_cols:
    df[col] = df[col].apply(lambda x: introduction_text + x if pd.notnull(x) else x)

# Save updated CSV (overwrite or create new file)
df.to_csv(csv_file, index=False, encoding="utf-8")

print(f"Added introduction to columns {question_cols} for all rows in {csv_file}.")
print(df.head(3))  # preview first 3 rows
